# 📱 WhatsApp Chat Analyzer
### Apni chat ka hisaab-kitaab — kaun kitna bolta hai

**Author:** Muhammad Ali Hassan  
Google Colab pe ready-to-run notebook.

---

### Kaise use karein?
1. Runtime → Run all
2. Apna WhatsApp chat export (`.txt`) upload karo
3. Results automatically generate ho jayenge

## 1. Install Required Libraries

In [ ]:
!pip install -q wordcloud emoji seaborn

## 2. Import Libraries

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import emoji
from datetime import datetime
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ Libraries loaded successfully!')

## 3. Upload WhatsApp Chat File

In [ ]:
print('📤 Apna WhatsApp chat export file (.txt) upload karo...')
uploaded = files.upload()

chat_file = list(uploaded.keys())[0]
print(f'\n✅ File uploaded: {chat_file}')

## 4. Parse Chat Data

In [ ]:
def parse_whatsapp_chat(file_path):
    """
    Parse WhatsApp exported chat (.txt)
    Supports common Android & iOS formats
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Common patterns
    patterns = [
        # Android: 12/11/2023, 10:30 AM - Name: Message
        r'(\d{1,2}/\d{1,2}/\d{2,4}),?\s+(\d{1,2}:\d{2}(?::\d{2})?\s*(?:AM|PM|am|pm)?)\s+-\s+([^:]+):\s+(.*)',
        # Android 24h: 12/11/2023, 22:30 - Name: Message
        r'(\d{1,2}/\d{1,2}/\d{2,4}),?\s+(\d{1,2}:\d{2}(?::\d{2})?)\s+-\s+([^:]+):\s+(.*)',
        # iOS: [12/11/2023, 10:30:45 AM] Name: Message
        r'\[(\d{1,2}/\d{1,2}/\d{2,4}),?\s+(\d{1,2}:\d{2}(?::\d{2})?\s*(?:AM|PM|am|pm)?)\]\s+([^:]+):\s+(.*)',
        # iOS 24h
        r'\[(\d{1,2}/\d{1,2}/\d{2,4}),?\s+(\d{1,2}:\d{2}(?::\d{2})?)\]\s+([^:]+):\s+(.*)',
        # Another common format
        r'(\d{1,2}\.\d{1,2}\.\d{2,4}),?\s+(\d{1,2}:\d{2}(?::\d{2})?\s*(?:AM|PM|am|pm)?)\s+-\s+([^:]+):\s+(.*)',
    ]

    messages = []
    lines = content.split('\n')
    current_msg = None

    for line in lines:
        line = line.strip()
        if not line:
            continue

        matched = False
        for pattern in patterns:
            match = re.match(pattern, line)
            if match:
                date_str, time_str, sender, message = match.groups()
                sender = sender.strip()
                message = message.strip()

                # Combine date + time
                datetime_str = f'{date_str} {time_str}'
                messages.append({
                    'datetime_str': datetime_str,
                    'sender': sender,
                    'message': message
                })
                matched = True
                break

        if not matched and messages:
            # Multi-line message continuation
            messages[-1]['message'] += ' ' + line

    df = pd.DataFrame(messages)

    if df.empty:
        raise ValueError('No messages found. Please check the file format.')

    # Try parsing datetime
    def try_parse_date(s):
        formats = [
            '%d/%m/%Y %I:%M %p', '%d/%m/%Y %H:%M',
            '%d/%m/%y %I:%M %p', '%d/%m/%y %H:%M',
            '%m/%d/%Y %I:%M %p', '%m/%d/%Y %H:%M',
            '%m/%d/%y %I:%M %p', '%m/%d/%y %H:%M',
            '%d/%m/%Y %I:%M:%S %p', '%d/%m/%Y %H:%M:%S',
            '%d.%m.%Y %H:%M', '%d.%m.%y %H:%M',
            '%d/%m/%Y, %I:%M %p', '%d/%m/%Y, %H:%M',
            '%m/%d/%Y, %I:%M %p', '%m/%d/%Y, %H:%M',
        ]
        for fmt in formats:
            try:
                return datetime.strptime(s.strip(), fmt)
            except:
                continue
        return None

    df['datetime'] = df['datetime_str'].apply(try_parse_date)
    df = df.dropna(subset=['datetime']).reset_index(drop=True)

    # Extra columns
    df['date'] = df['datetime'].dt.date
    df['year'] = df['datetime'].dt.year
    df['month'] = df['datetime'].dt.month_name()
    df['day'] = df['datetime'].dt.day_name()
    df['hour'] = df['datetime'].dt.hour
    df['minute'] = df['datetime'].dt.minute
    df['word_count'] = df['message'].apply(lambda x: len(str(x).split()))
    df['char_count'] = df['message'].apply(lambda x: len(str(x)))

    # Media detection
    media_keywords = ['<Media omitted>', 'image omitted', 'video omitted',
                      'audio omitted', 'sticker omitted', 'GIF omitted',
                      'document omitted', 'Contact card omitted']
    df['is_media'] = df['message'].apply(
        lambda x: any(k.lower() in str(x).lower() for k in media_keywords)
    )

    # Deleted messages
    df['is_deleted'] = df['message'].str.contains(
        'This message was deleted|You deleted this message|message deleted', 
        case=False, na=False
    )

    return df

# Parse the uploaded file
df = parse_whatsapp_chat(chat_file)
print(f'✅ Successfully parsed {len(df)} messages!')
print(f'📅 Date range: {df["datetime"].min().date()} → {df["datetime"].max().date()}')
print(f'👥 Participants: {df["sender"].nunique()}')
df.head()

## 5. Overall Statistics

In [ ]:
print('=' * 50)
print('📊 OVERALL CHAT STATISTICS')
print('=' * 50)
print(f'Total Messages      : {len(df):,}')
print(f'Total Participants : {df["sender"].nunique()}')
print(f'Total Words        : {df["word_count"].sum():,}')
print(f'Total Characters   : {df["char_count"].sum():,}')
print(f'Media Messages     : {df["is_media"].sum():,}')
print(f'Deleted Messages   : {df["is_deleted"].sum():,}')
print(f'Date Range         : {df["datetime"].min().date()} to {df["datetime"].max().date()}')
print(f'Total Days         : {(df["datetime"].max() - df["datetime"].min()).days + 1}')
print('=' * 50)

## 6. Kaun Kitna Bolta Hai? (Message Count by Sender)

In [ ]:
sender_stats = df.groupby('sender').agg(
    messages=('message', 'count'),
    words=('word_count', 'sum'),
    media=('is_media', 'sum')
).sort_values('messages', ascending=False)

sender_stats['percentage'] = (sender_stats['messages'] / sender_stats['messages'].sum() * 100).round(1)

print('\n🏆 MESSAGE RANKING\n')
print(sender_stats.to_string())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = sns.color_palette('husl', len(sender_stats))
axes[0].barh(sender_stats.index[::-1], sender_stats['messages'][::-1], color=colors[::-1])
axes[0].set_xlabel('Number of Messages')
axes[0].set_title('Messages per Person', fontsize=14, fontweight='bold')

# Pie chart
axes[1].pie(sender_stats['messages'], labels=sender_stats.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Message Share %', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Activity by Day & Hour

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df['day'].value_counts().reindex(day_order)

sns.barplot(x=day_counts.index, y=day_counts.values, ax=axes[0], palette='viridis')
axes[0].set_title('Messages by Day of Week', fontsize=14, fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

# Hour of day
hour_counts = df['hour'].value_counts().sort_index()
axes[1].plot(hour_counts.index, hour_counts.values, marker='o', linewidth=2, color='#e74c3c')
axes[1].fill_between(hour_counts.index, hour_counts.values, alpha=0.3, color='#e74c3c')
axes[1].set_title('Messages by Hour of Day', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Hour (0-23)')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

print(f'\n🔥 Most Active Day  : {day_counts.idxmax()} ({day_counts.max()} messages)')
print(f'🔥 Most Active Hour : {hour_counts.idxmax()}:00 ({hour_counts.max()} messages)')

## 8. Heatmap (Day × Hour)

In [ ]:
heatmap_data = df.groupby(['day', 'hour']).size().unstack(fill_value=0)
heatmap_data = heatmap_data.reindex(day_order)

plt.figure(figsize=(16, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, fmt='d', linewidths=0.5)
plt.title('Activity Heatmap (Day × Hour)', fontsize=16, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

## 9. Daily Timeline

In [ ]:
daily = df.groupby('date').size()

plt.figure(figsize=(16, 5))
plt.plot(daily.index, daily.values, color='#3498db', linewidth=1.5)
plt.fill_between(daily.index, daily.values, alpha=0.3, color='#3498db')
plt.title('Daily Message Volume', fontsize=16, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Messages')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f'📈 Peak Day: {daily.idxmax()} with {daily.max()} messages')

## 10. Word Cloud & Top Words

In [ ]:
# Combine all text messages (exclude media)
text_messages = df[~df['is_media']]['message'].astype(str)
all_text = ' '.join(text_messages)

# Simple stopwords (English + Hindi common)
stopwords = set([
    'the', 'to', 'and', 'a', 'of', 'in', 'is', 'it', 'you', 'that', 'for',
    'on', 'are', 'with', 'as', 'this', 'was', 'but', 'be', 'have', 'from',
    'or', 'one', 'had', 'by', 'not', 'what', 'all', 'were', 'we', 'when',
    'your', 'can', 'said', 'there', 'use', 'an', 'each', 'which', 'she',
    'do', 'how', 'their', 'if', 'will', 'up', 'other', 'about', 'out',
    'many', 'then', 'them', 'these', 'so', 'some', 'her', 'would', 'make',
    'like', 'him', 'into', 'time', 'has', 'look', 'two', 'more', 'write',
    'go', 'see', 'number', 'no', 'way', 'could', 'people', 'my', 'than',
    'first', 'water', 'been', 'call', 'who', 'oil', 'its', 'now', 'find',
    'long', 'down', 'day', 'did', 'get', 'come', 'made', 'may', 'part',
    'hai', 'ho', 'hain', 'ka', 'ke', 'ki', 'ko', 'se', 'par', 'bhi',
    'nahi', 'nahin', 'main', 'mein', 'ye', 'yeh', 'woh', 'kya', 'kyu',
    'kyun', 'aur', 'toh', 'to', 'hi', 'tha', 'thi', 'the', 'raha',
    'rahe', 'rahi', 'gaya', 'gayi', 'gaye', 'media', 'omitted', 'message',
    'deleted', 'this', 'was'
])

words = re.findall(r'\b[a-zA-Z\u0900-\u097F]{3,}\b', all_text.lower())
filtered_words = [w for w in words if w not in stopwords]

word_freq = Counter(filtered_words)
top_words = word_freq.most_common(20)

print('\n🔤 TOP 20 WORDS\n')
for i, (word, count) in enumerate(top_words, 1):
    print(f'{i:2d}. {word:15s} → {count}')

# Word Cloud
wc = WordCloud(
    width=1200, height=500,
    background_color='white',
    max_words=150,
    colormap='viridis',
    collocations=False
).generate_from_frequencies(word_freq)

plt.figure(figsize=(16, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud', fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 11. Emoji Analysis

In [ ]:
def extract_emojis(text):
    return [c for c in str(text) if c in emoji.EMOJI_DATA]

all_emojis = []
for msg in df['message']:
    all_emojis.extend(extract_emojis(msg))

if all_emojis:
    emoji_freq = Counter(all_emojis)
    top_emojis = emoji_freq.most_common(15)

    print('\n😊 TOP EMOJIS\n')
    for i, (em, count) in enumerate(top_emojis, 1):
        print(f'{i:2d}. {em}  →  {count}')

    # Plot
    emojis_list = [e[0] for e in top_emojis]
    counts_list = [e[1] for e in top_emojis]

    plt.figure(figsize=(12, 5))
    plt.bar(range(len(emojis_list)), counts_list, color=sns.color_palette('pastel', len(emojis_list)))
    plt.xticks(range(len(emojis_list)), emojis_list, fontsize=16)
    plt.title('Most Used Emojis', fontsize=14, fontweight='bold')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No emojis found in the chat.')

## 12. Per Person Deep Dive

In [ ]:
print('\n📋 DETAILED STATS PER PERSON\n')
print('=' * 70)

for sender in df['sender'].unique():
    person = df[df['sender'] == sender]
    avg_words = person['word_count'].mean()
    media_count = person['is_media'].sum()
    print(f'\n👤 {sender}')
    print(f'   Messages     : {len(person):,}')
    print(f'   Total Words  : {person["word_count"].sum():,}')
    print(f'   Avg Words/Msg: {avg_words:.1f}')
    print(f'   Media Shared : {media_count}')
    print(f'   First Msg    : {person["datetime"].min()}')
    print(f'   Last Msg     : {person["datetime"].max()}')

print('\n' + '=' * 70)

## ✅ Done!

Aapka WhatsApp chat analysis complete ho gaya.  
Agar aapko koi feature add karna ho toh batao!

---
**Made by Muhammad Ali Hassan**  
*Apni chat ka hisaab-kitaab — kaun kitna bolta hai*